# [13-2강] RNN forward 흐름과 출력 shape - 실습

In [2]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. batch_first 설정에 맞게 입력 바꾸기

batch_first=False인 RNN은 `[seq_len, batch, input_size]` 형태를 받습니다.

In [3]:
x_batch_first = torch.randn(3, 5, 4)  # [batch, seq, input]
rnn = nn.RNN(input_size=4, hidden_size=6, batch_first=False)
# TODO: [seq, batch, input] 형태로 바꾸세요.
x_time_first = x_batch_first.permute(1, 0, 2)
try:
    out, h = rnn(x_time_first)
    print(out.shape, h.shape)
except RuntimeError as e:
    print('입력 shape를 다시 확인하세요:', str(e).split('\\n')[0])


torch.Size([5, 3, 6]) torch.Size([1, 3, 6])


## 문제 2. 마지막 time step output 선택하기

RNN output에서 마지막 time step에 해당하는 hidden state를 선택합니다. shape뿐 아니라 단층·단방향 RNN에서 `h_n[-1]`과 값까지 같은지 확인합니다.

In [4]:
x = torch.randn(4, 6, 3)
rnn = nn.RNN(input_size=3, hidden_size=5, batch_first=True)
out, h_n = rnn(x)
# TODO: 마지막 time step output을 선택하세요.
last_output = out[:, -1, :]
print('last_output:', last_output.shape)
print('h_n[-1]:', h_n[-1].shape)
print('same values:', torch.allclose(last_output, h_n[-1]))


last_output: torch.Size([4, 5])
h_n[-1]: torch.Size([4, 5])
same values: True


## 문제 3. num_layers가 있는 RNN hidden shape 해석하기

RNN layer 수를 2로 늘린 뒤 마지막 layer의 hidden을 선택합니다. h_n의 첫 차원이 2인지, 선택한 값이 `h_n[-1]`과 같은지 함께 확인합니다.

In [9]:
x = torch.randn(4, 6, 3)
# TODO: 2-layer RNN으로 수정하세요.
rnn = nn.RNN(3, 5, num_layers=2, batch_first=True)
classifier = nn.Linear(5, 2)
out, h_n = rnn(x)
# TODO: 마지막 layer hidden을 선택하세요.
last_layer_hidden = h_n[-1]
logits = classifier(last_layer_hidden)
print('h_n:', h_n.shape)
print('logits:', logits.shape)
print('uses last layer:', h_n.shape[0] == 2 and torch.allclose(last_layer_hidden, h_n[-1]))


h_n: torch.Size([2, 4, 5])
logits: torch.Size([4, 2])
uses last layer: True


### 해설 및 실행 결과 해석

- h_n의 첫 번째 차원은 layer 수입니다. 여러 layer를 쌓은 경우 마지막 layer의 hidden을 사용하는 것이 최종 sequence 표현에 가깝습니다.